# Run NRLMF in Noisy Case

In [1]:
import os
import time
import gzip
import pickle
import warnings

%matplotlib inline
import matplotlib.pyplot as plt

import numpy as np

from tqdm import TqdmSynchronisationWarning
warnings.simplefilter("ignore", TqdmSynchronisationWarning)

In [2]:
PATH_ROOT = "/Users/sijianfan/Documents/projects/BiSSGL"
PATH_DATA = os.path.join(PATH_ROOT, "datasets/realAnalysis/tuberculosis/cv_data")

PATH_OUTPUT = os.path.join(PATH_ROOT, "outputs/results/realAnalysis/tuberculosis/noisy")
if not os.path.isdir(PATH_OUTPUT):
    os.makedirs(PATH_OUTPUT)

PATH_ARCHIVE = os.path.join(PATH_OUTPUT, "archived")
if not os.path.isdir(PATH_ARCHIVE):
    os.makedirs(PATH_ARCHIVE)

In [3]:
filename_staged = os.path.join(PATH_DATA, "staged_dataset.gz")

filenames = {"input": "staged_dataset.gz", "output": "results_nrlmf.gz"}

In [4]:
filename_input = os.path.join(PATH_DATA, filenames["input"])

filename_output = os.path.join(PATH_OUTPUT, filenames["output"])

if os.path.exists(filename_output):
    mdttm = time.strftime("%Y%m%d_%H%M%S")
    os.rename(
        filename_output,
        os.path.join(PATH_ARCHIVE, "%s%s" % (mdttm, filenames["output"])),
    )

In [5]:
from sgimc.utils import mc_split

In [6]:
from sgimc.utils import get_submatrix

In [7]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
from sgimc.utils import sparsify_with_mask


def mc_get_scores(R_true, R_prob):
    R_pred = np.where(R_prob.data > 0.5, 1, -1)

    # compute the confusion matrix for ±1 labels (`-1` is negative)
    ii, jj = ((R_pred + 1) // 2).astype(int), ((R_true.data + 1) // 2).astype(int)
    cnfsn = confusion_matrix(y_true=jj, y_pred=ii)

    return {
        "tn": cnfsn[0, 0],
        "fn": cnfsn[1, 0],
        "fp": cnfsn[0, 1],
        "tp": cnfsn[1, 1],
        "auc": roc_auc_score(R_true.data, R_prob.data),
    }

In [8]:
random_state = np.random.RandomState(0x0BADCAFE)

## Import model

In [10]:
from PyDTI3.nrlmf import NRLMF

## Load data

In [11]:
from sgimc.utils import load, save

U, V, Y = load(filename_input)

Y = Y.astype(float)

## Without selection

### Prepare similarity matrix

In [18]:
# Quick prepare of similarity matrix
from sklearn.metrics.pairwise import cosine_similarity

simD = cosine_similarity(U[:, :])
simT = cosine_similarity(V[:, :])

In [131]:
# Jaccard distance → convert to similarity
from sklearn.metrics import pairwise_distances

simD = 1 - pairwise_distances(U.toarray(), metric="jaccard")
simT = 1 - pairwise_distances(V.toarray(), metric="jaccard")

/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2182: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)
/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2182: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


In [21]:
# Just use no information
simD = np.eye(U.shape[0])
simT = np.eye(V.shape[0])

In [13]:
simD.shape, simT.shape

((6949, 6949), (13, 13))

### Implementation

In [22]:
dvlp_size, test_size = 0.9, 0.1

ind_dvlp, ind_test = next(
    mc_split(
        Y,
        n_splits=1,
        random_state=random_state,
        train_size=dvlp_size,
        test_size=test_size,
    )
)

Y_test = get_submatrix(Y, ind_test)

In [23]:
from sklearn.model_selection import ParameterGrid

grid_dataset = ParameterGrid(
    {
        "train_size": np.arange(0.001, 0.2, 0.02),
        # "train_size": [0.2],
        "n_splits": [2],
    }
)

grid_model = ParameterGrid(
    {
        "c": [1, 3, 5, 7, 9],  # confidence level
        # "c": [5],  # confidence level
        "K1": [5],
        "K2": [5],
        "r": [50],  # rank
        "lambda_d": [0.125],
        "lambda_t": [0.125],
        "alpha": [0.25],
        "beta": [0.125],
        "theta": [0.5],
        "max_iter": [100],
    }
)

In [24]:
from tqdm import tqdm
from sklearn.model_selection import ShuffleSplit, KFold
from sklearn.model_selection import train_test_split
from scipy.special import expit


results = []
for par_dtst in tqdm(grid_dataset):
    # prepare the train dataset: take the specified share from the beginnig of the index array
    ind_train_all, _ = train_test_split(
        ind_dvlp,
        shuffle=False,
        random_state=random_state,
        test_size=(1 - (par_dtst["train_size"] / dvlp_size)),
    )

    # Run the experiment: the model
    for par_mdl in grid_model:  # tqdm.tqdm(, desc="cv %02d" % (cv,))
        Y_train = get_submatrix(Y, ind_train_all)

        # Y_train[Y_train == 0] = 0.5
        Y_train[Y_train == -1] = 0.0

        # set up the model
        model = NRLMF(
            cfix=par_mdl["c"],
            K1=par_mdl["K1"],
            K2=par_mdl["K2"],
            num_factors=par_mdl["r"],
            lambda_d=par_mdl["lambda_d"],
            lambda_t=par_mdl["lambda_t"],
            alpha=par_mdl["alpha"],
            beta=par_mdl["beta"],
            theta=par_mdl["theta"],
            max_iter=par_mdl["max_iter"],
        )

        # fit on the whole development dataset
        model.fix_model(
            np.ones((Y_train.shape[0], Y_train.shape[1])), Y_train.toarray(), simD, simT
        )
        est_A, est_B = model.U, model.V

        # get the score
        prob_full = expit(simD @ est_A @ est_B.T @ simT.T)
        prob_test = get_submatrix(prob_full, ind_test)
        scores_test = mc_get_scores(Y_test, prob_test)
        d1_test = sum(abs(est_A).max(axis=1) > 0)
        d2_test = sum(abs(est_B).max(axis=1) > 0)

        # record the results
        results.append(
            {
                "train_size": par_dtst["train_size"],
                "lambda_d": par_mdl["lambda_d"],
                "lambda_t": par_mdl["lambda_t"],
                "c": par_mdl["c"],
                "r": par_mdl["r"],
                "test_score": scores_test["auc"],
                "test_d1": d1_test,
                "test_d2": d2_test,
            }
        )

100%|██████████| 10/10 [06:00<00:00, 36.02s/it]


In [25]:
results

[{'train_size': 0.001,
  'lambda_d': 0.125,
  'lambda_t': 0.125,
  'c': 1,
  'r': 50,
  'test_score': 0.3765061808922311,
  'test_d1': 6949,
  'test_d2': 13},
 {'train_size': 0.001,
  'lambda_d': 0.125,
  'lambda_t': 0.125,
  'c': 3,
  'r': 50,
  'test_score': 0.5400407982790087,
  'test_d1': 6949,
  'test_d2': 13},
 {'train_size': 0.001,
  'lambda_d': 0.125,
  'lambda_t': 0.125,
  'c': 5,
  'r': 50,
  'test_score': 0.7124923954014915,
  'test_d1': 6949,
  'test_d2': 13},
 {'train_size': 0.001,
  'lambda_d': 0.125,
  'lambda_t': 0.125,
  'c': 7,
  'r': 50,
  'test_score': 0.4512319972236682,
  'test_d1': 6949,
  'test_d2': 13},
 {'train_size': 0.001,
  'lambda_d': 0.125,
  'lambda_t': 0.125,
  'c': 9,
  'r': 50,
  'test_score': 0.39536537613232736,
  'test_d1': 6949,
  'test_d2': 13},
 {'train_size': 0.021,
  'lambda_d': 0.125,
  'lambda_t': 0.125,
  'c': 1,
  'r': 50,
  'test_score': 0.5244659533501,
  'test_d1': 6949,
  'test_d2': 13},
 {'train_size': 0.021,
  'lambda_d': 0.125,
  'l

In [26]:
filename_output

'/Users/sijianfan/Documents/projects/BiSSGL/outputs/results/realAnalysis/tuberculosis/noisy/results_nrlmf.gz'

In [27]:
with gzip.open(filename_output, "wb+", 4) as fout:
    pickle.dump(results, fout)

---

## With selection

In [28]:
from sgimc import SparseGroupIMCClassifier

C_lasso, C_group, C_ridge, rank = (0.0, 1.0, 1.0, 13)

imc = SparseGroupIMCClassifier(
    rank,
    n_threads=8,
    random_state=42,
    C_lasso=C_lasso,
    C_group=C_group,
    C_ridge=C_ridge,
)

# fit on the whole development dataset
Y_train = get_submatrix(Y, ind_train_all)
imc.fit(U, V, Y_train)

# get the score
prob_full = imc.predict_proba(U, V)
prob_test = get_submatrix(prob_full, ind_test)
scores_test = mc_get_scores(Y_test, prob_test)
d1_test = sum(abs(imc.coef_W_).max(axis=1) > 0)
d2_test = sum(abs(imc.coef_H_).max(axis=1) > 0)

print(scores_test, d1_test, d2_test)

{'tn': 5300, 'fn': 402, 'fp': 344, 'tp': 2988, 'auc': 0.9666375810373196} 1459 22


### Prepare similarity matrix

In [29]:
# Jaccard distance → convert to similarity
from sklearn.metrics import pairwise_distances

simD = 1 - pairwise_distances(
    U[:, np.where(abs(imc.coef_W_).max(axis=1) > 0)[0]].toarray(), metric="jaccard"
)
simT = 1 - pairwise_distances(
    V[:, np.where(abs(imc.coef_H_).max(axis=1) > 0)[0]].toarray(), metric="jaccard"
)

/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2182: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)
/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2182: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


### Implementation

In [30]:
dvlp_size, test_size = 0.9, 0.1

ind_dvlp, ind_test = next(
    mc_split(
        Y,
        n_splits=1,
        random_state=random_state,
        train_size=dvlp_size,
        test_size=test_size,
    )
)

Y_test = get_submatrix(Y, ind_test)

In [31]:
from sklearn.model_selection import ParameterGrid

grid_dataset = ParameterGrid(
    {
        "train_size": np.arange(0.001, 0.2, 0.02),
        # "train_size": [0.2],
        "n_splits": [2],
    }
)

grid_model = ParameterGrid(
    {
        "c": [1, 3, 5, 7, 9],  # confidence level
        # "c": [5],  # confidence level
        "K1": [5],
        "K2": [5],
        "r": [50],  # rank
        "lambda_d": [0.125],
        "lambda_t": [0.125],
        "alpha": [0.25],
        "beta": [0.125],
        "theta": [0.5],
        "max_iter": [100],
    }
)

In [ ]:
from tqdm import tqdm
from sklearn.model_selection import ShuffleSplit, KFold
from sklearn.model_selection import train_test_split
from scipy.special import expit


results = []
for par_dtst in tqdm(grid_dataset):
    # prepare the train dataset: take the specified share from the beginnig of the index array
    ind_train_all, _ = train_test_split(
        ind_dvlp,
        shuffle=False,
        random_state=random_state,
        test_size=(1 - (par_dtst["train_size"] / dvlp_size)),
    )

    # Run the experiment: the model
    for par_mdl in grid_model:  # tqdm.tqdm(, desc="cv %02d" % (cv,))
        Y_train = get_submatrix(Y, ind_train_all)

        # Y_train[Y_train == 0] = 0.5
        Y_train[Y_train == -1] = 0.0

        # set up the model
        model = NRLMF(
            cfix=par_mdl["c"],
            K1=par_mdl["K1"],
            K2=par_mdl["K2"],
            num_factors=par_mdl["r"],
            lambda_d=par_mdl["lambda_d"],
            lambda_t=par_mdl["lambda_t"],
            alpha=par_mdl["alpha"],
            beta=par_mdl["beta"],
            theta=par_mdl["theta"],
            max_iter=par_mdl["max_iter"],
        )

        # fit on the whole development dataset
        model.fix_model(
            np.ones((Y_train.shape[0], Y_train.shape[1])), Y_train.toarray(), simD, simT
        )
        est_A, est_B = model.U, model.V

        # get the score
        prob_full = expit(simD @ est_A @ est_B.T @ simT.T)
        prob_test = get_submatrix(prob_full, ind_test)
        scores_test = mc_get_scores(Y_test, prob_test)
        d1_test = sum(abs(est_A).max(axis=1) > 0)
        d2_test = sum(abs(est_B).max(axis=1) > 0)

        # record the results
        results.append(
            {
                "train_size": par_dtst["train_size"],
                "lambda_d": par_mdl["lambda_d"],
                "lambda_t": par_mdl["lambda_t"],
                "c": par_mdl["c"],
                "r": par_mdl["r"],
                "test_score": scores_test["auc"],
                "test_d1": d1_test,
                "test_d2": d2_test,
            }
        )

  0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
results

[{'train_size': 0.001,
  'lambda_d': 0.125,
  'lambda_t': 0.125,
  'c': 1,
  'r': 50,
  'test_score': 0.3765061808922311,
  'test_d1': 6949,
  'test_d2': 13},
 {'train_size': 0.001,
  'lambda_d': 0.125,
  'lambda_t': 0.125,
  'c': 3,
  'r': 50,
  'test_score': 0.5400407982790087,
  'test_d1': 6949,
  'test_d2': 13},
 {'train_size': 0.001,
  'lambda_d': 0.125,
  'lambda_t': 0.125,
  'c': 5,
  'r': 50,
  'test_score': 0.7124923954014915,
  'test_d1': 6949,
  'test_d2': 13},
 {'train_size': 0.001,
  'lambda_d': 0.125,
  'lambda_t': 0.125,
  'c': 7,
  'r': 50,
  'test_score': 0.4512319972236682,
  'test_d1': 6949,
  'test_d2': 13},
 {'train_size': 0.001,
  'lambda_d': 0.125,
  'lambda_t': 0.125,
  'c': 9,
  'r': 50,
  'test_score': 0.39536537613232736,
  'test_d1': 6949,
  'test_d2': 13},
 {'train_size': 0.021,
  'lambda_d': 0.125,
  'lambda_t': 0.125,
  'c': 1,
  'r': 50,
  'test_score': 0.5244659533501,
  'test_d1': 6949,
  'test_d2': 13},
 {'train_size': 0.021,
  'lambda_d': 0.125,
  'l

In [193]:
filename_output

'/Users/sijianfan/Documents/projects/BiSSGL/outputs/results/realAnalysis/tuberculosis/results_drimc.gz'

In [194]:
with gzip.open(filename_output, "wb+", 4) as fout:
    pickle.dump(results, fout)